# # M08 - LoRA, QLoRA, and the cost of adaptation
#
# **Hardware:** CPU (about 45 seconds of training) · **Time:** 3-4 hours ·
# **Prerequisite:** M07
#
# Full fine-tuning updates every weight. LoRA freezes the base model and learns a
# small low-rank update per adapted layer. QLoRA adds a quantized base model so
# the frozen weights fit in less memory. This notebook measures all three on the
# same FAQ task: quality, trainable parameters, adapter size, merged size, and
# quantization error.
#
# The model is small, so the absolute numbers are modest. The *ratios* are the
# lesson: you can adapt 3% of the parameters and keep most of the quality, and you
# can cut the weight footprint by 4-7x while barely moving the predictions.
#
# **You will produce:** a LoRA adapter, a merged model, two quantized copies, and
# a trade-off table.

# ## 1. Setup and reference models
#
# To compare adaptation methods you need a fixed baseline. We rebuild the same
# base model and the same full fine-tune as M07, with the same seeds. That costs
# about 30 seconds and removes the most common excuse for a comparison: "the
# baseline was trained differently".
#
# The task is the Northwind FAQ: 120 training pairs, 16 held-out questions that
# are disjoint by question template but share topics with training.

In [1]:
from pathlib import Path
import copy
import sys
import time

import torch

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "projects").is_dir())
sys.path.insert(0, str(ROOT / "projects" / "llm-fine-tuning"))

import llm_core as lc

lc.ensure_datasets(quiet=True)
train_pairs, test_pairs = lc.load_faq("train"), lc.load_faq("test")
faq_text = "\n".join(f"{row['question']}\n{row['answer']}" for row in train_pairs)
tokenizer = lc.CharTokenizer.from_text(lc.load_corpus() + "\n" + faq_text + "\nUser: Assistant:\n")

base = lc.build_model(len(tokenizer), block_size=160, n_layer=2, n_head=4, n_embd=96, dropout=0.1)
started = time.perf_counter()
base_history = lc.train_lm(
    base, tokenizer.encode(faq_text), steps=400, batch_size=8, block_size=160, lr=3e-3,
    seed=1, warmup=20, log_every=200,
)
print("base", base_history.summary(), "wall", round(time.perf_counter() - started, 1), "s")

train_examples = lc.build_examples(train_pairs, tokenizer, block_size=160)
test_examples = lc.build_examples(test_pairs, tokenizer, block_size=160)

full_sft = copy.deepcopy(base)
started = time.perf_counter()
lc.train_sft(
    full_sft, train_examples, steps=120, batch_size=8, lr=5e-4, seed=2,
    pad_id=tokenizer.pad_id, warmup=10, log_every=60,
)
print("full SFT wall", round(time.perf_counter() - started, 1), "s")


def quality(model):
    return {
        "answer_loss": round(lc.estimate_sft_loss(model, test_examples, pad_id=tokenizer.pad_id), 3),
        "accuracy": round(lc.masked_accuracy(model, test_examples, pad_id=tokenizer.pad_id), 3),
        "first_12": round(lc.masked_accuracy(model, test_examples, pad_id=tokenizer.pad_id, first_k=12), 3),
    }

print("full SFT quality:", quality(full_sft))
print("base quality    :", quality(base))

base {'steps': 400, 'first_loss': 4.0392, 'last_loss': 1.2014, 'seconds': 17.16, 'tokens': 512000, 'tokens_per_second': 29832, 'evals': []} wall 18.0 s


full SFT wall 5.4 s
full SFT quality: {'answer_loss': 0.315, 'accuracy': 0.905, 'first_12': 0.818}
base quality    : {'answer_loss': 1.039, 'accuracy': 0.688, 'first_12': 0.411}


# ## 2. LoRA in one equation
#
# For a frozen weight matrix `W` of shape `d_out x d_in`, LoRA learns
#
# ```text
# W' = W + (alpha / r) * B @ A
# ```
#
# where `A` is `r x d_in` and `B` is `d_out x r`. The adapter has
# `r * (d_in + d_out)` parameters instead of `d_in * d_out`. `B` is initialised to
# zero, so the adapted model starts exactly equal to the base model. `alpha / r`
# keeps the update scale roughly constant when you change the rank.
#
# We adapt the attention output projection (`c_attn`) and both `c_proj` layers in
# each block - six matrices in this two-layer model. Everything else is frozen.

In [2]:
lora_model = copy.deepcopy(base)
before = copy.deepcopy(lora_model)
before.eval()

report = lc.apply_lora(lora_model, rank=4, alpha=8.0)
print("adapted layers:", len(report["replaced"]))
for name in report["replaced"]:
    print("  ", name)
print(lc.format_table([
    {"item": "trainable parameters", "value": report["trainable_parameters"]},
    {"item": "total parameters (with adapters)", "value": report["total_parameters"]},
    {"item": "trainable fraction", "value": report["trainable_fraction"]},
    {"item": "adapter bytes", "value": lc.lora_bytes(lora_model)},
], ["item", "value"]))

# Initial identity check: B is zero, so logits must not move.
lora_model.eval()
x, _ = lc.pad_batch(test_examples[:4], tokenizer.pad_id)
with torch.no_grad():
    logits_before, _ = before(x)
    logits_initial, _ = lora_model(x)
max_delta = float((logits_before - logits_initial).abs().max())
print("max logit change at initialisation:", max_delta)
assert max_delta < 1e-7

adapted layers: 6
   blocks.0.attn.c_attn
   blocks.0.attn.c_proj
   blocks.0.mlp.c_proj
   blocks.1.attn.c_attn
   blocks.1.attn.c_proj
   blocks.1.mlp.c_proj
item                              value 
--------------------------------  ------
trainable parameters              8448  
total parameters (with adapters)  253056
trainable fraction                0.0334
adapter bytes                     33792 
max logit change at initialisation: 0.0


# ## 3. Train the adapter
#
# Only the adapter weights receive gradients. The optimizer state is therefore
# proportional to the adapter size, not to the model size - the second half of
# LoRA's memory advantage. Note the higher learning rate: with so few trainable
# parameters, a larger step is normal.

In [3]:
started = time.perf_counter()
lora_history = lc.train_sft(
    lora_model, train_examples, steps=150, batch_size=8, lr=1e-3, seed=3,
    pad_id=tokenizer.pad_id, warmup=10, log_every=50,
)
print(lora_history.summary(), "wall", round(time.perf_counter() - started, 1), "s")
print("LoRA quality:", quality(lora_model))

rows = [
    {"method": "full SFT", **quality(full_sft), "trainable": full_sft.parameter_count(), "artifact": "model"},
    {"method": "LoRA r=4", **quality(lora_model), "trainable": report["trainable_parameters"], "artifact": lc.lora_bytes(lora_model)},
]
print(lc.format_table(rows, ["method", "answer_loss", "accuracy", "first_12", "trainable", "artifact"]))
print(f"trainable reduction: {full_sft.parameter_count() / report['trainable_parameters']:.1f}x")
assert quality(lora_model)["answer_loss"] < quality(base)["answer_loss"]

{'steps': 150, 'first_loss': 1.2848, 'last_loss': 0.911, 'seconds': 6.51, 'tokens': 118706, 'tokens_per_second': 18223, 'evals': []} wall 6.5 s
LoRA quality: {'answer_loss': 0.601, 'accuracy': 0.816, 'first_12': 0.708}


method    answer_loss  accuracy  first_12  trainable  artifact
--------  -----------  --------  --------  ---------  --------
full SFT  0.315        0.905     0.818     244608     model   
LoRA r=4  0.601        0.816     0.708     8448       33792   
trainable reduction: 29.0x


# ## 4. Merge the adapter
#
# For serving, you can fold `B @ A` into `W` and ship an ordinary model with no
# adapter code. Merging is exact arithmetic, not an approximation: the cell below
# checks that logits before and after merging agree to floating-point tolerance.
#
# The trade-off: a merged model is one artifact per task. If you serve ten
# fine-tunes of the same base, keeping ten small adapters and swapping them at
# request time is far cheaper than storing ten full copies.

In [4]:
lora_model.eval()
with torch.no_grad():
    logits_trained, _ = lora_model(x)

merged = copy.deepcopy(lora_model)
merged_count = lc.merge_lora(merged)
merged.eval()
with torch.no_grad():
    logits_merged, _ = merged(x)

print("merged layers:", merged_count)
print("max logit difference:", float((logits_trained - logits_merged).abs().max()))
print("parameters after merge:", merged.parameter_count())
print("float32 bytes after merge:", merged.param_bytes(4), "vs adapter", lc.lora_bytes(lora_model))
assert merged_count == len(report["replaced"])
assert torch.allclose(logits_trained, logits_merged, atol=1e-5)

merged layers:

 6
max logit difference: 6.67572021484375e-06
parameters after merge: 244608
float32 bytes after merge: 978432 vs adapter 33792


# ## 5. Weight-only quantization
#
# Quantization rounds weights onto a coarser grid: 8-bit uses 256 levels per
# output row, 4-bit uses 16. The scale per row is stored in float32. This
# notebook simulates the *accuracy and size* effects by dequantizing before the
# forward pass.
#
# **What is not measured here: speed.** Without int8/int4 kernels the arithmetic
# is still float32, so any latency number would be meaningless. The tables report
# size and quality only.

In [5]:
full_sft.eval()
with torch.no_grad():
    logits_full, _ = full_sft(x)

quant_rows = []
for bits in (8, 4):
    clone, q = lc.quantized_copy(full_sft, bits)
    clone.eval()
    with torch.no_grad():
        logits_q, _ = clone(x)
    metrics = quality(clone)
    quant_rows.append({
        "bits": bits,
        "size_kib": round(q["quantized_bytes"] / 1024, 1),
        "compression": q["compression_ratio"],
        "max_error": q["max_abs_error"],
        "accuracy": metrics["accuracy"],
        "top1_agreement": round(lc.top1_agreement(logits_full, logits_q), 3),
    })

print(lc.format_table(quant_rows, ["bits", "size_kib", "compression", "max_error", "accuracy", "top1_agreement"]))
print("float32 size KiB:", round(full_sft.param_bytes(4) / 1024, 1))
assert quant_rows[1]["size_kib"] < quant_rows[0]["size_kib"]
assert quant_rows[1]["max_error"] >= quant_rows[0]["max_error"]
assert quant_rows[0]["top1_agreement"] > 0.9

bits  size_kib  compression  max_error  accuracy  top1_agreement
----  --------  -----------  ---------  --------  --------------
8     259.8     3.758        0.001282   0.905     0.992         
4     139.1     7.022        0.023086   0.9       0.895         


float32 size KiB: 955.5


# ## 6. The memory arithmetic behind QLoRA
#
# QLoRA keeps the base weights quantized (usually 4-bit NF4 with double
# quantization) and trains LoRA adapters in higher precision. This notebook does
# not implement NF4 or paged optimizers; the table below is parameter-memory
# arithmetic under stated assumptions, not a measurement of a GPU run.
#
# Assumptions: `P` parameter values, adapters have `A` values, full fine-tuning
# stores weights + gradients + two Adam moments (4 bytes each = 16 bytes per
# parameter), LoRA stores frozen base weights plus adapter, gradient and two
# moments (16 bytes per adapter value). Activations and temporary buffers are
# excluded - at realistic batch sizes they dominate, and they are exactly what
# gradient checkpointing addresses.

In [6]:
P = full_sft.parameter_count()
A = report["trainable_parameters"]

def mib(values: int) -> float:
    return round(values / (1024 ** 2), 3)

memory_rows = [
    {"strategy": "full fine-tune (fp32 + Adam)", "weights": mib(4 * P), "optimizer + grads": mib(12 * P), "total": mib(16 * P)},
    {"strategy": "LoRA on fp32 base", "weights": mib(4 * P), "optimizer + grads": mib(16 * A), "total": mib(4 * P + 16 * A)},
    {"strategy": "QLoRA-style int8 base", "weights": mib(P + 4 * (P // 96)), "optimizer + grads": mib(16 * A), "total": mib(P + 4 * (P // 96) + 16 * A)},
    {"strategy": "QLoRA-style int4 base", "weights": mib(P // 2 + 4 * (P // 96)), "optimizer + grads": mib(16 * A), "total": mib(P // 2 + 4 * (P // 96) + 16 * A)},
]
print(lc.format_table(memory_rows, ["strategy", "weights", "optimizer + grads", "total"]))
print("\nMiB, parameter memory only; scales approximated as one float32 per output row.")

strategy                      weights  optimizer + grads  total
----------------------------  -------  -----------------  -----
full fine-tune (fp32 + Adam)  0.933    2.799              3.732
LoRA on fp32 base             0.933    0.129              1.062
QLoRA-style int8 base         0.243    0.129              0.372
QLoRA-style int4 base         0.126    0.129              0.255

MiB, parameter memory only; scales approximated as one float32 per output row.


# ## 7. Which configuration would you ship?
#
# There is no universal answer, but the comparison narrows it:
#
# | Requirement | Reasonable choice |
# | --- | --- |
# | one task, smallest latency | merged model, quantized if the quality budget allows |
# | many tasks, one base | unmerged adapters, swapped per request |
# | tight memory for training | QLoRA-style quantized base + adapters |
# | maximum quality, memory available | full fine-tune |
#
# The measured table above is the evidence. Note what it does **not** say: 4-bit
# preserved accuracy here because the task and model are tiny. On a larger model
# and a harder task, 4-bit degradation is real and must be measured on your own
# evaluation set.

In [7]:
# A compact decision record for the artifact you would ship.
decision = {
    "base_parameters": P,
    "adapter_parameters": A,
    "adapter_fraction": report["trainable_fraction"],
    "adapter_kib": round(lc.lora_bytes(lora_model) / 1024, 1),
    "int8_kib": quant_rows[0]["size_kib"],
    "int4_kib": quant_rows[1]["size_kib"],
    "quality_delta_int4_vs_fp32": round(quant_rows[1]["accuracy"] - quality(full_sft)["accuracy"], 3),
}
print(decision)

{'base_parameters': 244608, 'adapter_parameters': 8448, 'adapter_fraction': 0.0334, 'adapter_kib': 33.0, 'int8_kib': 259.8, 'int4_kib': 139.1, 'quality_delta_int4_vs_fp32': -0.005}


# ## 8. Validation and exercises
#
# **Exercises**
#
# 1. Try rank 1, 2, 8, and 16. Plot adapter size against held-out accuracy. Where
#    does the curve flatten?
# 2. Adapt only `c_attn` instead of six layers. How much quality is lost?
# 3. Merge the adapter and then quantize the merged model. Compare quality with
#    quantizing before merging. Does order matter?
# 4. Increase SFT steps to 400. Does LoRA close the gap with full fine-tuning, or
#    does it overfit first?
# 5. Write the adapter plus a base digest into a manifest, then verify it with
#    `eval_core.artifacts.verify_manifest` from M16. Why is a digest necessary?

In [8]:
checks = {
    "adapter starts as identity": max_delta < 1e-7,
    "adapter trained": quality(lora_model)["answer_loss"] < quality(base)["answer_loss"],
    "few trainable parameters": report["trainable_fraction"] < 0.1,
    "merge is exact": torch.allclose(logits_trained, logits_merged, atol=1e-5),
    "int8 is smaller than float32": quant_rows[0]["size_kib"] < full_sft.param_bytes(4) / 1024,
    "int4 is smaller than int8": quant_rows[1]["size_kib"] < quant_rows[0]["size_kib"],
    "int8 preserves predictions": quant_rows[0]["top1_agreement"] > 0.9,
    "adapter artifact is small": lc.lora_bytes(lora_model) < full_sft.param_bytes(4) / 10,
}
for name, passed in checks.items():
    print(f"{'ok' if passed else 'FAIL'} - {name}")
    assert passed, name
print("\nM08 validation passed.")

ok - adapter starts as identity
ok - adapter trained
ok - few trainable parameters
ok - merge is exact
ok - int8 is smaller than float32
ok - int4 is smaller than int8
ok - int8 preserves predictions
ok - adapter artifact is small

M08 validation passed.
